In [1]:
import sys
import os
import h5py
from pathlib import Path
import time
import random

SRC = Path.cwd().parent
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from helpers.config import *
from model_training.model import *

import torch
from torch.utils.data import Dataset, Subset, random_split, DataLoader
import torchvision.models as models
import torch.nn as nn
from sklearn.metrics import confusion_matrix
import optuna
from model_training.model import CNN, LISADataset

import warnings
warnings.filterwarnings("ignore")

/opt/anaconda3/envs/gwaves_lisa_py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

In [3]:
DEVICE = torch.device("mps")

def compute_loss(model, loader, criterion):
    model.eval()
    loss = 0.0
    count = 0
    with torch.no_grad():
        for batch in loader:
            images, labels = batch[0].to(DEVICE), batch[1].to(DEVICE)
            outputs = model(images)
            loss += float(criterion(outputs, labels))
            count += 1
    loss /= count
    return loss

def train_model(model, train_data, val_data,
                learning_rate, batch_size,
                num_epochs):
    model.train()
    model = model.to(DEVICE)
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size)
    
    criterion = torch.nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

    for e in range(num_epochs):
        for batch in train_loader:
            images, labels = batch[0].to(DEVICE), batch[1].to(DEVICE)

            z = model(images)
            loss = criterion(z, labels)

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

    valid_loss = compute_loss(model, val_loader, criterion)

    return valid_loss


In [4]:
dataset = LISADataset(training_hm_dataset_path)

unique_indices = np.unique(dataset.sim_indices)
all_indices = np.array(dataset.sim_indices)

rng = np.random.default_rng(42)
rng.shuffle(unique_indices)

n_total = len(unique_indices)
n_train = int(0.7 * n_total)
n_val   = int(0.2 * n_total)

train_sources = unique_indices[:n_train]
val_sources   = unique_indices[n_train:n_train+n_val]
test_sources  = unique_indices[n_train+n_val:]

train_indices = np.where(np.isin(all_indices, train_sources))[0]
val_indices   = np.where(np.isin(all_indices, val_sources))[0]
test_indices  = np.where(np.isin(all_indices, test_sources))[0]

train_dataset = Subset(dataset, train_indices)
val_dataset   = Subset(dataset, val_indices)
test_dataset  = Subset(dataset, test_indices)

In [5]:
def objective(trial):
    set_seed(42)

    # ---- sample hyperparameters ----
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    width = trial.suggest_categorical("width", [4, 6, 8, 10])
    
    # ---- build model ----
    model = CNN(width=width, bn=True, normalise=False, drop=0.0)
    
    # ---- train (short run!) ----
    valid_loss = train_model(
        model,
        train_dataset,
        val_dataset,
        learning_rate=lr,
        batch_size=64,
        num_epochs=3
    )
    
    return valid_loss   # final validation loss

In [6]:
study = optuna.create_study(direction="minimize")
optuna.logging.set_verbosity(optuna.logging.INFO)
study.optimize(objective, n_trials=10)

[I 2026-04-10 23:04:21,314] A new study created in memory with name: no-name-b338f323-3486-4e99-8109-430a0dbce30f
[I 2026-04-10 23:11:15,128] Trial 0 finished with value: 0.1394635039337334 and parameters: {'lr': 0.0021833338205551308, 'width': 10}. Best is trial 0 with value: 0.1394635039337334.
[I 2026-04-10 23:18:04,377] Trial 1 finished with value: 0.2367053051434812 and parameters: {'lr': 0.005268570742413202, 'width': 10}. Best is trial 0 with value: 0.1394635039337334.
[I 2026-04-10 23:24:49,702] Trial 2 finished with value: 0.14610479905136994 and parameters: {'lr': 0.00036860995533746473, 'width': 6}. Best is trial 0 with value: 0.1394635039337334.
[I 2026-04-10 23:31:31,246] Trial 3 finished with value: 0.14163421544556817 and parameters: {'lr': 0.00016732558275781094, 'width': 8}. Best is trial 0 with value: 0.1394635039337334.
[I 2026-04-11 00:07:50,939] Trial 4 finished with value: 0.13436193081239858 and parameters: {'lr': 0.004632800531891599, 'width': 6}. Best is trial 

In [7]:
study.best_params

{'lr': 0.004632800531891599, 'width': 6}